# 14. Combined Algorithm: UCT + BestFirst Warm Restart

This notebook demonstrates how to combine multiple search algorithms on the **same tree** to improve retrosynthetic planning. Instead of discarding the search tree and starting over, we perform a *warm restart*: swap the algorithm (e.g. UCT → BestFirst) while keeping all expanded nodes and discovered paths.

## 1. Why Combine Algorithms?

UCT and BestFirst explore the search tree in fundamentally different ways:

| Property | UCT | BestFirst |
|----------|-----|-----------|
| **Selection** | UCB score: Q(s) + c·√N(parent)/(N(s)+1) | Globally sorted by evaluation score |
| **Exploration** | Balances exploitation vs exploration via visit counts | Always expands the single best-scored node |
| **Strength** | Discovers diverse paths, avoids local optima | Rapidly deepens the most promising branch |
| **Weakness** | May spread too thin across many branches | Can get stuck in a single subtree |

By running UCT first, we build a broad tree with many partially explored branches. Then BestFirst *re-prioritizes* those branches by evaluation score and rapidly drives the most promising ones to completion. 

## 2. Environment Setup

In [26]:
import os

os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = ""

import gc
import json
import time
from pathlib import Path

import yaml

## 3. Load Resources and Configure Policy

We load building blocks, reaction rules, and the combined policy network. See [Tutorial 08](08_Combined_Ranking_Filtering_Policy.ipynb) for details on the combined policy.

In [27]:
from synplan.utils.config import PolicyNetworkConfig, RDKitEvaluationConfig
from synplan.utils.loading import (
    download_preset,
    load_building_blocks,
    load_evaluation_function,
    load_reaction_rules,
)

paths = download_preset("synplanner-article", save_to=Path("synplan_data"))

building_blocks = load_building_blocks(
    paths["building_blocks"], standardize=True, silent=True
)
reaction_rules = load_reaction_rules(paths["reaction_rules"])

print(f"Building blocks: {len(building_blocks):,}")
print(f"Reaction rules:  {len(reaction_rules):,}")

Couldn't access the Hub to check for update but local file already exists. Defaulting to existing file. (error: [Errno -3] Temporary failure in name resolution)


Building blocks: 186,030
Reaction rules:  24,094


In [28]:
# ── Hyperparameters ──────────────────────
mcts_params = {
    "c_ucb": 0.1,              # exploration constant for UCT
    "max_iterations": 100000,  # virtually unreachable in 10 minutes
    "max_depth": 15,           # maximum tree depth (= maximum number of retrosynthetic steps)
    "evaluation_agg": "max",
}

policy_params = {
    "top_rules": 50,            # number of top-ranked rules to consider
    "filtering_threshold": 0.01, # hard veto threshold (0.0 = no veto)
    "temperature": 1.0,         # softmax temperature for ranking
}

print("Hyperparameters set:")
print(f"  c_ucb              : {mcts_params['c_ucb']}")
print(f"  top_rules          : {policy_params['top_rules']}")
print(f"  filtering_threshold: {policy_params['filtering_threshold']:.2e}")

Hyperparameters set:
  c_ucb              : 0.1
  top_rules          : 50
  filtering_threshold: 1.00e-02


### Optional: load per-bin Bayesian-optimized hyperparameters

If you are running the SA-score benchmark, each bin has its own tuned config in `optuna_config/heavy_atoms/`. Run the cell below to override the manual values above.

In [ ]:
# ── (Optional) Load from per-bin Bayesian-optimized config ─────
# Uncomment and run this cell to override the manual values above.

# BIN_LABEL = "3.5_4.5"
#
# config_path = Path(f"optuna_config/heavy_atoms/best_config_{BIN_LABEL}.yaml")
# with open(config_path) as f:
#     raw = yaml.safe_load(f)
#
# mcts_params = raw["mcts_config"]
# policy_params = raw["policy_config"]
#
# print(f"Loaded config for bin {BIN_LABEL}:")
# print(f"  c_ucb              : {mcts_params['c_ucb']:.6f}")
# print(f"  top_rules          : {policy_params['top_rules']}")
# print(f"  filtering_threshold: {policy_params['filtering_threshold']:.2e}")

In [29]:
from synplan.mcts.expansion import CombinedPolicyNetworkFunction

expansion_fn = CombinedPolicyNetworkFunction(
    filtering_config=PolicyNetworkConfig(
        weights_path=str(paths["filtering_policy"]),
        policy_type="filtering",
    ),
    ranking_config=PolicyNetworkConfig(
        weights_path=str(paths["ranking_policy"]),
        policy_type="ranking",
    ),
    top_rules=policy_params["top_rules"],
    rule_prob_threshold=0.0,
    temperature=policy_params.get("temperature", 1.0),
    filtering_threshold=policy_params["filtering_threshold"],
)

print("Combined policy loaded.")

Lightning automatically upgraded your loaded checkpoint from v1.9.5 to v2.6.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint synplan_data/policy/supervised_gcn/v1/v1/filtering_policy.ckpt`
Lightning automatically upgraded your loaded checkpoint from v1.9.5 to v2.6.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint synplan_data/policy/supervised_gcn/v1/v1/ranking_policy.ckpt`


Combined policy loaded.


### Load evaluation functions

As found with the heuristics benchmark (see notebook 15), simple molecular descriptors can drastically help navigate the search.
Here, we will pre-load the `heavyAtomCount` heuristic, but many others are available in SynPlanner and may be best suited for specific purposes.

In [37]:
eval_fn = load_evaluation_function(RDKitEvaluationConfig(score_function="heavyAtomCount"))

## 4. Phase 1: UCT Search (Cold Start)

Phase 1 builds the initial tree from scratch using UCT. UCT balances exploration and exploitation via the Upper Confidence Bound:

$$\text{UCB}(s) = Q(s) + c \cdot \frac{\sqrt{N(\text{parent})}}{N(s) + 1}$$

where $Q(s)$ is the node's estimated value, $N$ are visit counts, and $c$ (`c_ucb`) balances exploration and exploitation.

In [38]:
from synplan.chem.utils import mol_from_smiles
from synplan.mcts.tree import Tree
from synplan.utils.config import TreeConfig

# ── Target molecule (paste any SMILES here) ────────────────────
target_smiles = "CCC(C)N1C(=O)C(=CNNc2nc3c(c(=O)n(C)c(=O)n3C)n2C)C(=O)NC1=S"

# ── Or load from a benchmark file ──────────────────────────────
# BIN_LABEL = "3.5_4.5"
# smi_file = Path(f"/home/YOUR_PATH/benchmarks/sascore/targets_with_sascore_{BIN_LABEL}.smi")
# smiles_list = [l.strip().split()[0] for l in smi_file.read_text().splitlines() if l.strip()]
# target_smiles = smiles_list[0]

mol = mol_from_smiles(target_smiles, standardize=True, clean2d=True, clean_stereo=True)
print(f"Target: {target_smiles}")
print(f"Atoms:  {len(mol)}")
mol  # display 2D structure

Target: CCC(C)N1C(=O)C(=CNNc2nc3c(c(=O)n(C)c(=O)n3C)n2C)C(=O)NC1=S
Atoms:  30


In [39]:
P1_TIME = 120  # seconds

p1_config = TreeConfig(
    algorithm="uct",
    search_strategy="evaluation_first",
    max_iterations=mcts_params.get("max_iterations", 100000),
    max_time=P1_TIME,
    max_depth=mcts_params.get("max_depth", 15),
    min_mol_size=6,
    init_node_value=0.5,
    ucb_type="uct",
    c_ucb=mcts_params["c_ucb"],
    evaluation_agg=mcts_params.get("evaluation_agg", "max"),
    silent=True,
)

print(f"Phase 1 config:")
print(f"  Algorithm:   UCT")
print(f"  Time budget: {P1_TIME}s")
print(f"  c_ucb:       {mcts_params['c_ucb']:.6f}")

Phase 1 config:
  Algorithm:   UCT
  Time budget: 120s
  c_ucb:       0.100000


In [40]:
tree = Tree(
    target=mol,
    config=p1_config,
    reaction_rules=reaction_rules,
    building_blocks=building_blocks,
    expansion_function=expansion_fn,
    evaluation_function=eval_fn,
)

t0 = time.time()
solved_p1 = False
for s, _ in tree:
    if s:
        solved_p1 = True
        break
t_p1 = time.time() - t0

print(f"\nPhase 1 result: {'SOLVED' if solved_p1 else 'NOT SOLVED'}")
print(f"  Time:       {t_p1:.1f}s")
print(f"  Iterations: {tree.curr_iteration}")
print(f"  Expanded:   {len(tree.expanded_nodes)} nodes")
print(f"  Total:      {len(tree.nodes)} nodes")
print(f"  Routes:     {len(tree.winning_nodes)}")


Phase 1 result: NOT SOLVED
  Time:       120.4s
  Iterations: 156
  Expanded:   156 nodes
  Total:      5160 nodes
  Routes:     0


## 5. Inspecting the Tree Between Phases

Before running Phase 2, let's inspect the tree state. The key insight is that UCT has expanded many nodes and partially explored many branches. Some of these branches are close to completion but UCT moved on due to its exploration policy.

In [41]:
# Count unexpanded nodes (candidates for BestFirst's frontier)
unexpanded = set()
for nid in tree.nodes:
    if nid not in tree.expanded_nodes and not tree.nodes[nid].is_solved():
        unexpanded.add(nid)

# Depth distribution
depths = {}
for nid, depth in tree.nodes_depth.items():
    depths[depth] = depths.get(depth, 0) + 1

print(f"Tree state after Phase 1:")
print(f"  Total nodes:      {len(tree.nodes)}")
print(f"  Expanded:         {len(tree.expanded_nodes)}")
print(f"  Unexpanded:       {len(unexpanded)} (BestFirst frontier candidates)")
print(f"  Solved nodes:     {sum(1 for n in tree.nodes.values() if n.is_solved())}")
print(f"\n  Depth distribution:")
for d in sorted(depths.keys()):
    print(f"    depth {d:2d}: {depths[d]:5d} nodes")

Tree state after Phase 1:
  Total nodes:      5160
  Expanded:         156
  Unexpanded:       5004 (BestFirst frontier candidates)
  Solved nodes:     0

  Depth distribution:
    depth  0:     1 nodes
    depth  1:    35 nodes
    depth  2:  1107 nodes
    depth  3:  3867 nodes
    depth  4:   150 nodes


## 6. Warm Restart: UCT → BestFirst

A warm restart swaps the search algorithm on an existing tree without rebuilding it. The steps are:

1. **Reset counters**: clear iteration count, timer, and winning nodes so the new phase runs fresh.
2. **Create new algorithm**: instantiate BestFirst (or any other algorithm).
3. **Seed the frontier**: for BestFirst, we scan all unexpanded children of expanded nodes and insert them into the priority frontier, scored by the evaluation function.
4. **Continue iterating**: the tree's `__next__` method now delegates to the new algorithm.

BestFirst immediately has a rich frontier to work with, rather than starting from a single root node.

Note: running this cell will destroy winning nodes from Phase 1

In [44]:
from synplan.mcts.algorithm import UCT, BestFirst


def warm_restart_search(tree, algorithm_name, eval_fn, max_time, params=None):
    """Swap algorithm on an existing tree and continue searching.

    Args:
        tree: Existing Tree object with expanded nodes.
        algorithm_name: "best_first", "uct", "breadth_first", or "beam"
        (see synplan/mcts/algorithm.py for details).
        eval_fn: Evaluation function for this phase (can differ from Phase 1).
        max_time: Time budget in seconds.
        params: Optional dict to override c_ucb, epsilon, evaluation_agg, ...

    Returns:
        (tree, solved, elapsed)
    """
    algo_map = {"uct": UCT, "best_first": BestFirst}
    algo_cls = algo_map.get(algorithm_name)
    if algo_cls is None:
        raise ValueError(f"Unknown algorithm: {algorithm_name}")

    # Step 1: Swap evaluator if provided
    if eval_fn is not None:
        tree.evaluator = eval_fn

    # Step 2: Update config for this phase
    tree.config.max_time = max_time
    tree.config.max_iterations = (params or {}).get("max_iterations", 100000)
    if params:
        if "c_ucb" in params:
            tree.config.c_ucb = params["c_ucb"]
        if "epsilon" in params:
            tree.config.epsilon = params["epsilon"]
        if "evaluation_agg" in params:
            tree.config.evaluation_agg = params["evaluation_agg"]

    # Step 3: Reset counters so the new phase runs for its full time budget
    tree.curr_iteration = 0
    tree.curr_time = 0
    tree.found_a_route = False
    tree.winning_nodes.clear()

    # Step 4: Create new algorithm instance
    new_algo = algo_cls(tree)

    # Step 5: Seed BestFirst frontier from existing unexpanded nodes
    if algorithm_name == "best_first" and hasattr(new_algo, "frontier"):
        for nid in tree.nodes:
            if nid in tree.expanded_nodes:
                for cid in tree.children.get(nid, set()):
                    if cid not in tree.expanded_nodes and not tree.nodes[cid].is_solved():
                        score = tree._get_node_value(cid)
                        depth = tree.nodes_depth.get(cid, 0)
                        new_algo.insert_sorted_frontier(cid, score, depth, False)
        print(f"  Seeded BestFirst frontier with {len(new_algo.frontier)} nodes")

    tree.algorithm = new_algo

    # Step 6: Run the search
    tree.start_time = time.time()
    tree._tqdm = True
    solved = False
    try:
        for s, _ in tree:
            if s:
                solved = True
                break
    except StopIteration:
        pass
    except Exception as e:
        print(f"  Warm restart error: {e}")
    elapsed = time.time() - tree.start_time
    return tree, solved, elapsed

### Run Phase 2: BestFirst on Phase 1's tree

BestFirst re-prioritizes all the frontier nodes UCT discovered and expands them in strict best-score-first order. We can also widen the policy (`top_rules` increased, `filtering_threshold` disabled) to allow more reaction rules.

In [45]:
P2_TIME = 120  # seconds

if not solved_p1:
    # Widen the policy for BestFirst: more rules, no hard veto
    expansion_fn.top_rules = policy_params["top_rules"]
    expansion_fn.filtering_threshold = None

    expanded_before = len(tree.expanded_nodes)

    tree, solved_p2, t_p2 = warm_restart_search(
        tree, "best_first", eval_fn, P2_TIME
    )

    expanded_after = len(tree.expanded_nodes)

    print(f"\nPhase 2 (BestFirst) result: {'SOLVED' if solved_p2 else 'NOT SOLVED'}")
    print(f"  Time:          {t_p2:.1f}s")
    print(f"  New expanded:  {expanded_after - expanded_before} nodes")
    print(f"  Total expanded: {expanded_after} nodes")
    print(f"  Routes:        {len(tree.winning_nodes)}")
else:
    print("Already solved in Phase 1, skipping Phase 2.")

  Seeded BestFirst frontier with 5004 nodes

Phase 2 (BestFirst) result: SOLVED
  Time:          22.1s
  New expanded:  96 nodes
  Total expanded: 252 nodes
  Routes:        1


## 7. Optional: Save the tree for visualization 

See SynPlanner fork `visualization`

In [46]:
import pickle
from pathlib import Path

# Save the tree for the visualization scripts in tutorials/visualization/
# Usage:
#   python visualization/visualize_tree.py  --tree visualization/tree.pkl --out visualization/tree.html
#   python visualization/expansion_tree.py  --tree visualization/tree.pkl --output visualization/expansion_evol.html
tree_pkl = Path("visualization/tree.pkl")
tree_pkl.parent.mkdir(exist_ok=True)

tree._tqdm = None  # clear the tqdm progress bar — its internal lambda is not picklable

with open(tree_pkl, "wb") as f:
    pickle.dump(tree, f)

print(f"Tree saved to {tree_pkl.resolve()}")
print(f"  Nodes         : {len(tree)}")
print(f"  Winning nodes : {len(tree.winning_nodes)}")

Tree saved to /home/gschmitz/SynPlanner_v2/tutorials/visualization/tree.pkl
  Nodes         : 6789
  Winning nodes : 1


## 8. Optional: Extract the Route

If any phase found a solution, we extract the route graph from the tree's winning node.

In [54]:
from IPython.display import SVG, display
from synplan.utils.visualisation import extract_routes, get_route_svg_from_json

solved_any = solved_p1 or solved_p2 

if solved_any:
    phase = 1 if solved_p1 else 2 
    route_length = tree.nodes_depth[tree.winning_nodes[0]]
    print(f"Solved in Phase {phase}")
    print(f"Route has {route_length} retrosynthetic steps")

    routes_block = extract_routes(tree)
    routes_json  = {i: r for i, r in enumerate(routes_block)}
    svg_str = get_route_svg_from_json(routes_json, route_id=0)
    display(SVG(svg_str))
else:
    print("Not solved in any phase.")
    print(f"Final tree: {len(tree.expanded_nodes)} expanded, {len(tree.nodes)} total nodes")

Solved in Phase 2
Route has 15 retrosynthetic steps
